## Before running

This public copy has no course data, saved model, or review-level outputs. Obtain the course files from NTULearn, place them in `data/course/`, then run:

```sh
python train_course.py --output runs/course-20261013 --seed 42
```

Aggregate results from the completed experiment are in `results/course-20261013/`. Notebook outputs were cleared before publishing.

# IE4483: trained TF-IDF sentiment classifier

This notebook extends the lecturer's `tfidf_features_example.ipynb` using the actual Option 1 dataset. **Aggregate results from a completed training run are included.** Train locally to regenerate the saved model and review-level artifacts. Running this notebook reads saved results and lets you try sentences; full retraining is optional in the last cell. Labels: **0 = negative, 1 = positive**.

The lecturer example fits default unigram TF-IDF on training text and transforms test text with the same vectorizer. Here we add classifiers, class imbalance handling, word/character n-grams, duplicate handling, cross-validation, a separate holdout, model persistence and verified submission export.

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display, Image
from sentiment import classify_text

ROOT = Path.cwd()
RUN = ROOT / "runs" / "course-20261013"
DATA = ROOT / "data" / "course"
assert (RUN / "model.joblib").exists(), "Open this notebook from the sentiment-prototype folder."
audit = json.loads((RUN / "data_audit.json").read_text(encoding="utf-8"))
selected = json.loads((RUN / "selection.json").read_text(encoding="utf-8"))["selected_model"]
print("Selected:", selected)
display(pd.Series(audit, name="value").to_frame())

## 1. Data and evaluation design

The 7,401 supplied training rows contain 219 exact case/whitespace-normalised duplicates, leaving **7,182 unique reviews**. We reserve **1,437** for the final holdout and use **5,745** for three-fold model selection. Both labels occur in every partition. About 85% of reviews are positive, so we choose models using **macro-F1**, which gives both classes equal importance.

TF-IDF is fitted separately inside each development fold. The holdout is not used to choose settings. A fixed seed, saved splits and three fold scores support reproducibility. These are fold comparisons, not three independent seed runs. Exact duplicates are removed, but near-duplicate or related reviews may remain.

The test set has no labels. Its 101 reviews that match normalised training text are disclosed in the audit; predictions are still produced by the fitted model.

In [ ]:
splits = pd.read_csv(RUN / "split.csv")
display(pd.crosstab(splits["split"], splits["label"]).rename(columns={0: "negative", 1: "positive"}))
cv = pd.read_csv(RUN / "cv_results.csv")
display(cv[["model", "accuracy_mean", "macro_f1_mean", "macro_f1_std", "negative_recall_mean", "total_fit_seconds"]])
display(Image(filename=str(RUN / "model_comparison.png")))

## 2. Holdout performance

The winner was frozen in `selection.json` before holdout evaluation. We compare the selected model with the default TF-IDF + logistic baseline and the neural MLP. The majority-class baseline shows why high accuracy alone is insufficient. These scores describe the split-trained models; they are **not test-set scores** and do not separately evaluate the final model refit on all training data.

In [ ]:
metrics = json.loads((RUN / "holdout_metrics.json").read_text(encoding="utf-8"))
display(pd.DataFrame({name: {k: value[k] for k in ["accuracy", "macro_f1", "balanced_accuracy", "negative_recall", "positive_recall"]} for name, value in metrics.items()}).T)
display(Image(filename=str(RUN / "holdout_confusion.png")))

## 3. Inspect the final architecture

**Selected model:** review text → word TF-IDF (unigrams/bigrams) plus character TF-IDF (3–5 characters within words) → concatenated sparse vector → class-weighted logistic regression → binary sentiment. Both TF-IDF branches use sublinear term frequency and retain negation.

**Neural comparison:** word/phrase TF-IDF (up to 10,000 features) → Dense(64, ReLU) → one sigmoid output. Adam trains binary log loss with L2 regularisation. Internal early stopping uses only the training fold, with accuracy as its stopping metric. The MLP is not a recurrent or transformer model.

Inspect the saved metadata for exact settings. Weighted-model probabilities are estimates and have not been calibrated.

In [ ]:
artifact = joblib.load(RUN / "model.joblib")  # Load only your own/trusted model artifacts.
model = artifact["pipeline"]
feature_names = model.named_steps["tfidf"].get_feature_names_out()
print("Actual final feature dimension:", len(feature_names))
print("Classifier:", model.named_steps["classifier"])
print("Word/character branches:", [(name, len(vectorizer.vocabulary_)) for name, vectorizer in model.named_steps["tfidf"].transformer_list])
weights = pd.DataFrame({"feature": feature_names, "weight_toward_positive": model.named_steps["classifier"].coef_[0]})
display(pd.concat([weights.nsmallest(8, "weight_toward_positive"), weights.nlargest(8, "weight_toward_positive")]))

## 4. Examine successes and errors

These are labelled holdout examples from the model before final refitting. Use actual text to explain strengths and weaknesses in your report. A label disagreement alone does not prove that the supplied label is wrong. Do not repeatedly tune the model using these now-inspected holdout cases.

In [ ]:
examples = pd.read_csv(RUN / f"{selected}_holdout_predictions.csv")
with pd.option_context("display.max_colwidth", 240):
    for label in [0, 1]:
        for correct in [True, False]:
            print("Actual label:", label, "Correct:", correct)
            display(examples.loc[(examples.label == label) & (examples.correct == correct)].head(2))

## 5. Try your sentence or review

Change the text and rerun this cell. The final model uses all 7,182 unique labelled training reviews. The course corpus is domain-specific, so success on these reviews does not guarantee the same accuracy on hotel reviews or arbitrary sentences. Neutral/mixed reviews still receive a binary prediction.

In [ ]:
sentence = "The material is comfortable and the quality is excellent."
display(classify_text(RUN / "model.joblib", sentence))

## 6. Verify the completed submission

The actual course template uses **`reviews,predictions`**, not the generic `id,label` example in the briefing. The source template remains unchanged. Every original review and its row order must match `test.json`; predictions replace only the placeholder prediction column.

The generated file is `runs/course-20261013/submission.csv`. There is no test accuracy because test labels are withheld.

In [ ]:
test = json.loads((DATA / "test.json").read_text(encoding="utf-8"))
submission = pd.read_csv(RUN / "submission.csv", keep_default_na=False)
assert list(submission.columns) == ["reviews", "predictions"]
assert submission.reviews.tolist() == [r["reviews"] for r in test]
assert len(submission) == 1851
assert submission.predictions.isin([0, 1]).all()
print("Verified", len(submission), "rows with exact source review text/order.")
display(submission.predictions.value_counts().rename(index={0: "negative", 1: "positive"}).to_frame("predicted_count"))
display(submission.head())

## 7. Optional reproducible retraining

The completed results above are already saved. Set `RETRAIN = True` only to repeat the full experiment. A new run folder preserves the existing evidence. Retraining this frozen setup verifies reproducibility; use fresh nested evaluation if changing settings after inspecting the holdout. Consult `README.md` and `RESULTS.txt` for commands and report guidance.

In [ ]:
RETRAIN = False
if RETRAIN:
    import subprocess, sys
    subprocess.run([sys.executable, "train_course.py", "--seed", "42"], cwd=ROOT, check=True)
else:
    print("Using the completed saved run; no retraining requested.")